In [21]:
print("HELLO")

HELLO


In [6]:
from datasets import load_dataset

root_folders ='D:/PycharmProjects/TDFilter/DataSet/Biology/'
# 加载数据集
dataset = load_dataset('json', data_files="D:/PycharmProjects/TDFilter/DataSet/Biology/biologicExtend.json")
# 这里需要进行随机打乱
dataset = dataset.shuffle(seed=42)


In [7]:

from datasets import DatasetDict

# 首先将数据集划分为 80% 训练集，20% 剩余数据（用于划分验证集和测试集）
train_test_split = dataset['train'].train_test_split(test_size=0.90)

# 将剩余的 20% 再次划分为 50% 验证集和 50% 测试集，即每个 10% 数据
test_valid_split = train_test_split['test'].train_test_split(test_size=0.95)

# 创建新的 DatasetDict，包含训练集、验证集和测试集
dataset = DatasetDict({
    'train': train_test_split['train'],
    'valid': test_valid_split['train'],  # 10% 验证集
    'test': test_valid_split['test'],    # 10% 测试集
})

# 查看划分后的数据集
print(dataset)

# 将分好的训练集、验证集和测试集保存为文件
dataset['train'].to_json(root_folders+'train_dataset_10_5.json')
dataset['valid'].to_json(root_folders+'valid_dataset_10_5.json')
dataset['test'].to_json(root_folders+'test_dataset_10_5.json')



DatasetDict({
    train: Dataset({
        features: ['subtopic', 'generateKnowledge', 'flag', 'topic'],
        num_rows: 4467
    })
    valid: Dataset({
        features: ['subtopic', 'generateKnowledge', 'flag', 'topic'],
        num_rows: 2010
    })
    test: Dataset({
        features: ['subtopic', 'generateKnowledge', 'flag', 'topic'],
        num_rows: 38195
    })
})


Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/39 [00:00<?, ?ba/s]

10640404

TypeError: to_json() missing 1 required positional argument: 'path_or_buf'

In [8]:
from transformers import AutoTokenizer

# 加载 RoBERTa 分词器
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# 数据预处理函数
def preprocess_function(examples):
    # 对输入文本进行分词
    inputs = tokenizer(examples["generateKnowledge"], padding="max_length", truncation=True)
    # 将 flag 作为标签
    inputs["labels"] = examples["flag"]
    return inputs

# 对数据集进行分词处理
tokenized_datasets = dataset.map(preprocess_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/4467 [00:00<?, ? examples/s]

Map:   0%|          | 0/2010 [00:00<?, ? examples/s]

Map:   0%|          | 0/38195 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['subtopic', 'generateKnowledge', 'flag', 'topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4467
    })
    valid: Dataset({
        features: ['subtopic', 'generateKnowledge', 'flag', 'topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2010
    })
    test: Dataset({
        features: ['subtopic', 'generateKnowledge', 'flag', 'topic', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 38195
    })
})

In [9]:
from transformers import DataCollatorWithPadding

# 准备数据集用于训练
train_dataset = tokenized_datasets['train']
valid_dataset = tokenized_datasets['valid']
test_dataset = tokenized_datasets['test']
# 数据整理工具
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [10]:
# 定义标签
id2label = {0: "WRONG", 1: "RIGHT"}
label2id = {"WRONG": 0, "RIGHT": 1}

In [11]:
import evaluate

accuracy = evaluate.load("accuracy")

In [12]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [13]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 加载 BERT 模型进行二分类
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
import wandb
wandb.login()

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [15]:
training_args = TrainingArguments(
    output_dir="Biological_roberta_10_5",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_strategy="steps",
    logging_steps=100,
    push_to_hub=False,
    report_to="wandb",
)



In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [17]:
# 开始训练
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss


TrainOutput(global_step=280, training_loss=0.4247844696044922, metrics={'train_runtime': 544.5389, 'train_samples_per_second': 8.203, 'train_steps_per_second': 0.514, 'total_flos': 1175317084293120.0, 'train_loss': 0.4247844696044922, 'epoch': 1.0})

In [18]:
# 模型评估
trainer.evaluate(eval_dataset=test_dataset)

{'eval_loss': 0.30017220973968506,
 'eval_accuracy': 0.8799319282628616,
 'eval_runtime': 349.6195,
 'eval_samples_per_second': 109.247,
 'eval_steps_per_second': 6.83,
 'epoch': 1.0}